# AgriSwin-MoE: evaluate on Colab

Scores the runs **trained on Colab** on PlantVillage test and on the frozen
PlantDoc set, with MC-dropout uncertainty, then compares every run and redraws
the paper's figures. Runs trained on the laptop were evaluated there; their
results are already in `results/eval/`.

Any GPU will do: evaluation trains nothing, and takes a few minutes per run.

1. **Runtime > Change runtime type** > any GPU > Save.
2. **Runtime > Run all.**

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Connect Google Drive. Checkpoints and the image cache live there, so they
#    survive a disconnect. The first time, Colab asks you to allow access.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Get the latest code. In a runtime that has already trained, --rebase
#    keeps any result commits made here on top of the newest code.
import os
REPO = "/content/agriswin-moe"
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} pull --rebase
else:
    !git clone https://github.com/amritaj312-max/agriswin-moe.git {REPO}
%cd {REPO}
!git log --oneline -1

In [ ]:
# 4. Prepare this runtime: exact packages, images verified byte for byte
#    against the laptop's, checkpoints on Drive. The first time this downloads
#    the images (a few minutes) and caches them on Drive; after that it copies
#    the cache (about a minute).
import importlib
import sys
sys.path.insert(0, f"{REPO}/notebooks")
import colab_runtime
importlib.reload(colab_runtime)
runtime = colab_runtime.setup()

In [ ]:
# 5. Evaluate the runs that have weights on Drive and no evaluation yet
runs = colab_runtime.runs_needing_evaluation()
print("to evaluate:", ", ".join(runs) if runs else "nothing new")
RUNS = " ".join(runs)
if runs:
    !python ml_pipeline/src/evaluation/evaluate.py --runs {RUNS}

In [ ]:
# 6. Compare every run honestly (seed floor, then paired tests) and redraw the figures
!python ml_pipeline/src/evaluation/compare.py
!python ml_pipeline/src/evaluation/figures.py

In [ ]:
# 7. Save: commit and push the evaluations and figures (copied to Drive instead without a token)
colab_runtime.commit_and_push(
    ["results/eval", "paper/figures"],
    "Evaluate Colab runs: " + (", ".join(runs) if runs else "refresh comparison and figures"),
    runtime["can_push"])